# Baseline 


## Установка зависимостей

In [ ]:
!pip install pillow numpy scikit-image torch torchvision -q

## Конфигурация

In [10]:
import os, random, zipfile
import numpy as np
from PIL import Image

GRID     = 24
FS       = 20
IMG_SIZE = GRID * FS   # 480

# Укажите путь к датасету.
# Ожидаемая структура:
#   <DATA_DIR>/train/inputs/   — перемешанные изображения (480×480)
#   <DATA_DIR>/train/targets/  — оригинальные изображения (480×480)
#   <DATA_DIR>/test/           — тестовые изображения для сабмита
DATA_DIR = "/path/to/dataset"

## Шаг 1. Загрузка данных

In [ ]:
def load_image(path):
    return np.array(Image.open(path).convert("RGB"))

def extract_fragments(img):
    """480×480 → (576, 20, 20, 3)."""
    return np.array(
        [img[r * FS:(r + 1) * FS, c * FS:(c + 1) * FS]
         for r in range(GRID) for c in range(GRID)],
        dtype=np.uint8,
    )

def load_train(n=None):
    """Генератор (input_img, target_img, name)."""
    inp_dir = os.path.join(DATA_DIR, "train", "inputs")
    tgt_dir = os.path.join(DATA_DIR, "train", "targets")
    names = sorted(os.listdir(inp_dir))[:n]
    for name in names:
        yield (
            load_image(os.path.join(inp_dir, name)),
            load_image(os.path.join(tgt_dir, name)),
            name,
        )

def load_test():
    """Генератор (input_img, name)."""
    test_dir = os.path.join(DATA_DIR, "test")
    for name in sorted(os.listdir(test_dir)):
        yield load_image(os.path.join(test_dir, name)), name

inp, tgt, name = next(load_train(n=1))
print(f"Пример: {name}  input={inp.shape}  target={tgt.shape}")

### Просмотр нескольких изображений

In [ ]:
import matplotlib.pyplot as plt

samples = list(load_train(n=3))

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for i, (inp, tgt, name) in enumerate(samples):
    axes[i, 0].imshow(inp)
    axes[i, 0].set_title(f"{name}\nвход (перемешан)")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(tgt)
    axes[i, 1].set_title("оригинал")
    axes[i, 1].axis("off")
plt.tight_layout()
plt.show()

## Шаг 2. Baseline — простая CNN для сборки пазла

Идея: обучим маленькую сеть которая по паре фрагментов предсказывает, стоят ли они рядом (справа или снизу). Потом жадно соберём пазл используя эти предсказания.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class PairCNN(nn.Module):
    """Принимает пару фрагментов (20×40×3), предсказывает 3 класса:
       0 = не соседи, 1 = второй справа от первого, 2 = второй снизу от первого."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


model = PairCNN().to(device)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

### Подготовка обучающих пар

In [ ]:
class PairDataset(Dataset):
    """Генерирует пары фрагментов из оригинальных изображений (targets).
       Позитивные: настоящие соседи (справа / снизу).
       Негативные: случайные пары."""

    def __init__(self, data_dir, max_images=200):
        self.pairs = []
        tgt_dir = os.path.join(data_dir, "train", "targets")
        names = sorted(os.listdir(tgt_dir))[:max_images]

        for name in names:
            img = load_image(os.path.join(tgt_dir, name))
            frags = extract_fragments(img)  # (576, 20, 20, 3)

            for r in range(GRID):
                for c in range(GRID):
                    idx = r * GRID + c
                    # Пара "справа" (label=1)
                    if c < GRID - 1:
                        right = r * GRID + c + 1
                        pair = np.concatenate([frags[idx], frags[right]], axis=1)  # 20×40×3
                        self.pairs.append((pair, 1))
                    # Пара "снизу" (label=2)
                    if r < GRID - 1:
                        below = (r + 1) * GRID + c
                        pair = np.concatenate([frags[idx], frags[below]], axis=0)  # 40×20×3
                        pad = np.zeros((20, 40, 3), dtype=np.uint8)
                        pad[:, :20] = frags[idx]
                        pad[:, 20:] = frags[below]
                        self.pairs.append((pad, 2))

            # Негативные пары (случайные)
            n_neg = GRID * (GRID - 1) * 2  # столько же сколько позитивных
            for _ in range(n_neg // max_images + 1):
                i1, i2 = random.sample(range(len(frags)), 2)
                pair = np.concatenate([frags[i1], frags[i2]], axis=1)
                self.pairs.append((pair, 0))

        random.shuffle(self.pairs)
        print(f"Пар для обучения: {len(self.pairs):,}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair, label = self.pairs[idx]
        x = torch.from_numpy(pair).permute(2, 0, 1).float() / 255.0
        return x, label


dataset = PairDataset(DATA_DIR, max_images=200)
loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

### Обучение

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)

    acc = correct / total
    avg_loss = total_loss / total
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  acc={acc:.3f}")

### Сборка пазла с помощью CNN

Жадный алгоритм: для каждой позиции выбираем фрагмент, который CNN считает лучшим соседом.

In [ ]:
@torch.no_grad()
def solve_with_cnn(img, model, batch_size=512):
    """Жадная сборка пазла: предвычисляем все scores батчами, потом собираем."""
    model.eval()
    frags = extract_fragments(img)
    n = len(frags)

    # Предвычисляем scores для всех пар (i, j)
    right_scores = np.zeros((n, n), dtype=np.float32)
    below_scores = np.zeros((n, n), dtype=np.float32)

    for i in range(n):
        # Пары "справа": frags[i] | frags[j]
        pairs_r = [np.concatenate([frags[i], frags[j]], axis=1)
                    for j in range(n) if j != i]
        # Пары "снизу": frags[i] сверху, frags[j] снизу
        pairs_b = []
        for j in range(n):
            if j == i: continue
            pad = np.zeros((20, 40, 3), dtype=np.uint8)
            pad[:, :20] = frags[i]
            pad[:, 20:] = frags[j]
            pairs_b.append(pad)

        for pairs, scores_arr, label_idx in [(pairs_r, right_scores, 1),
                                              (pairs_b, below_scores, 2)]:
            x = torch.from_numpy(np.array(pairs)).permute(0, 3, 1, 2).float() / 255.0
            all_probs = []
            for start in range(0, len(x), batch_size):
                batch = x[start:start + batch_size].to(device)
                probs = torch.softmax(model(batch), dim=1).cpu().numpy()
                all_probs.append(probs)
            all_probs = np.concatenate(all_probs, axis=0)

            k = 0
            for j in range(n):
                if j == i: continue
                scores_arr[i][j] = all_probs[k, label_idx]
                k += 1

    # Жадная сборка по предвычисленным scores
    grid = [[None] * GRID for _ in range(GRID)]
    used = set()
    grid[0][0] = 0
    used.add(0)

    for r in range(GRID):
        for c in range(GRID):
            if r == 0 and c == 0:
                continue
            best_idx, best_score = None, -1
            for i in range(n):
                if i in used:
                    continue
                score, count = 0, 0
                if c > 0 and grid[r][c - 1] is not None:
                    score += right_scores[grid[r][c - 1]][i]
                    count += 1
                if r > 0 and grid[r - 1][c] is not None:
                    score += below_scores[grid[r - 1][c]][i]
                    count += 1
                if count:
                    score /= count
                if score > best_score:
                    best_score, best_idx = score, i
            grid[r][c] = best_idx
            used.add(best_idx)

    canvas = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for r in range(GRID):
        for c in range(GRID):
            canvas[r * FS:(r + 1) * FS, c * FS:(c + 1) * FS] = frags[grid[r][c]]
    return canvas

### Проверка на нескольких изображениях

In [ ]:
from skimage.metrics import structural_similarity as ssim

scores = []
for inp, tgt, name in load_train(n=3):
    result = solve_with_cnn(inp, model)
    s = ssim(tgt, result, channel_axis=2, data_range=255)
    scores.append(s)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(inp);    axes[0].set_title("Вход (перемешан)")
    axes[1].imshow(result); axes[1].set_title(f"CNN baseline\nSSIM={s:.4f}")
    axes[2].imshow(tgt);    axes[2].set_title("Оригинал")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"\nMean SSIM: {np.mean(scores):.4f}")

## Шаг 3. Генерация сабмита

In [ ]:
def make_submission(model, out_zip="submission.zip"):
    items = list(load_test())
    print(f"Генерирую сабмит: {len(items)} изображений → {out_zip}")

    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for i, (img, name) in enumerate(items):
            result = solve_with_cnn(img, model)
            img_id = os.path.splitext(name)[0]
            tmp = f"/tmp/{img_id}.png"
            Image.fromarray(result).save(tmp)
            zf.write(tmp, f"{img_id}.png")
            os.remove(tmp)
            if (i + 1) % 100 == 0 or i == 0:
                print(f"  {i + 1}/{len(items)}")

    print(f"Готово: {out_zip}")


make_submission(model)